In [ ]:
import pandas as pd
from datetime import datetime
import chardet
import pandas as pd
from haversine import haversine
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import pandas as pd
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:


df = pd.read_csv("seoul_with_address.csv")

def extract_city_gu_dong(addr):
    city, gu, dong = "", "", ""
    if pd.isna(addr):
        return city, gu, dong

    # 주소 문자열을 콤마로 분리
    parts = [p.strip() for p in addr.split(',')]

    for p in parts:
        if p.endswith('시'):
            city = p
        elif p.endswith('구'):
            gu = p
        elif p.endswith('동'):
            dong = p

    return city, gu, dong


df[['city', 'gu', 'dong']] = df['address'].apply(lambda x: pd.Series(extract_city_gu_dong(x)))

df.to_csv("seoul_with_city_gu_dong.csv", index=False, encoding='utf-8-sig')

print(df[['address', 'city', 'gu', 'dong']].head())


In [ ]:




print("📂 데이터 미리보기:")
print(df.head(3))
print()

# YYYYMM → datetime
df['buildDate'] = df['buildDate'].astype(str).str.zfill(6)  # '200006' 형태 유지
df['buildDate_dt'] = pd.to_datetime(df['buildDate'], format='%Y%m', errors='coerce')

# 현재 날짜 기준 (2025년 11월)
current = pd.Timestamp(year=2025, month=11, day=1)

# 경과 개월 / 년 계산
df['elapsed_months'] = (current.year - df['buildDate_dt'].dt.year) * 12 + \
                       (current.month - df['buildDate_dt'].dt.month)
df['elapsed_years'] = df['elapsed_months'] / 12

#건축경과년으로 대체
df['buildDate'] = df['elapsed_years']

# 중간 계산 열 제거
df = df.drop(columns=['id','min_sales','max_sales','buildDate_dt', 'elapsed_months', 'elapsed_years'], errors='ignore')


print("변환 완료")
print(df[['buildDate']].head())

df.to_csv("House.csv", index=False, encoding='utf-8-sig')
print("\n💾 저장 완료 → House_data_updated.csv")


<h1>여기까지가 시각화 전 데이터 전처리<h1>
<h1>여기서부터 1차 모델에 넣을 데이터 전처리<h1>

In [ ]:


missing_gu = df['gu'].isna() | (df['gu'].astype(str).str.strip() == "")

df.loc[missing_gu, 'gu'] = df.loc[missing_gu, 'city']


In [ ]:
df = df.dropna(subset=['avg_sales'])
df = df.drop(columns=['p'])

In [ ]:
drop_gus = ['구리시', '광명시', '의정부시','대구광역시']
df = df[~df['city'].isin(drop_gus)].reset_index(drop=True)

In [ ]:
df.isnull().sum()

In [ ]:
df

In [ ]:
df.to_csv("BaseData.csv", index=False, encoding='utf-8-sig')

<h1>여기서부터 2차 모델에 넣을 변수 추가<h1>
<h1>공공시설은 따로 받아옴<h1>

In [ ]:


houses = pd.read_csv("편의시설.csv")

stations = pd.read_csv("서울시 역사마스터 정보.csv", encoding="cp949")

station_lat_col = 'lat'   
station_lng_col = 'lng'   

# 가장 가까운 지하철역 거리 계산
def nearest_subway_dist(h_lat, h_lng):
    house_xy = (h_lat, h_lng)
    
    
    dists = stations.apply(
        lambda row: haversine(
            house_xy, 
            (row[station_lat_col], row[station_lng_col])
        ), 
        axis=1
    )
    return dists.min() 

# 4) 집 → 지하철역 최소거리 계산
houses["subway_nearest_km"] = houses.apply(
    lambda row: nearest_subway_dist(row["lat"], row["lng"]),
    axis=1
)

print("🎉 지하철 최소거리 계산 완료!")
print(houses[['lat','lng','subway_nearest_km']].head())


In [ ]:
df = houses

# =========================================
# 자치구별 가구증가율 붙이기
# =========================================

growth = pd.read_csv("2021_총가구증가율.csv")   # 필요하면 encoding='cp949' 옵션 추가


# 컬럼 이름 정리
growth = growth.rename(columns={
    'Category': 'gu',
    '서울특별시': 'household_growth_2021'   # 자치구별 총가구 증가율 (%)
})

#공백 제거
growth['gu'] = growth['gu'].str.strip()

df['gu'] = df['gu'].astype(str).str.strip()

# 머지
df = df.merge(growth[['gu', 'household_growth_2021']],
              on='gu', how='left')

# =========================================
# 자치구별 아파트 거래수 붙이기
# =========================================
apt = pd.read_csv("아파트매매거래+현황_20251130155056.csv")  

apt_clean = apt.iloc[1:].copy()

apt_clean = apt_clean[apt_clean['자치구별(1)'] == '합계'].copy()

# gu 이름 컬럼으로 재정의
apt_clean = apt_clean.rename(columns={
    '자치구별(2)': 'gu',
    '2021': 'apt_transactions_2021'   # 2021년 아파트 매매 거래수
})

# 공백 제거
apt_clean['gu'] = apt_clean['gu'].astype(str).str.strip()

apt_clean['apt_transactions_2021'] = pd.to_numeric(
    apt_clean['apt_transactions_2021'], errors='coerce'
)

# 필요한 컬럼만 남기기
apt_clean = apt_clean[['gu', 'apt_transactions_2021']]

# 메인 df에 머지
df = df.merge(apt_clean, on='gu', how='left')

# =========================================
# 결과 저장
# =========================================

print("✅ 병합 완료!")
print(df[['gu', 'household_growth_2021', 'apt_transactions_2021']].head())


In [ ]:
dong_target_mean = df.groupby('dong')['avg_sales'].mean()
df['dong_te'] = df['dong'].map(dong_target_mean)


In [ ]:
df.head(1)

In [ ]:
df.to_csv("useData.csv", index=False, encoding="utf-8-sig")

In [ ]:

df = pd.read_csv('useData.csv')
drop_df = df.drop(['lat','lng','address','city','gu','dong'], axis=1, inplace = False)
drop_df

# 1) 숫자형만 남기기
X_vif = drop_df.select_dtypes(include=[np.number]).copy()

# 2) inf → NaN
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)

# 3) NaN 제거
X_vif = X_vif.dropna()

In [ ]:

X = X_vif  # 수치형 변수만 넣기
X = add_constant(X)

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

In [ ]:

fig, ax = plt.subplots(figsize=(6, len(vif)*0.5 + 1))
ax.axis('off')

table = ax.table(
    cellText=vif.values,
    colLabels=vif.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.3)

plt.savefig("vif_table.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
df.head(1)

In [ ]:
df.isnull().sum()

In [ ]:
print(df['city'].value_counts())
sns.countplot(x = 'city', data = df)
plt.show()

In [ ]:
df = pd.read_csv('useData결측치수정.csv')
df.isnull().sum()

In [ ]:
df.to_csv("useData.csv", index=False, encoding="utf-8-sig")
df.isnull().sum()